# Phase 1A Results Visualization
Load JSON results from the phase1A folder and generate polished plots for analysis and reporting.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.dpi"] = 120

# Resolve base path (prefers workspace-relative path)
base_dir = Path.cwd() / "ouroboros" / "results" / "phase1A"
if not base_dir.exists():
    base_dir = Path.cwd()

output_dir = base_dir / "plots"
output_dir.mkdir(parents=True, exist_ok=True)

saved_files = []
base_dir, output_dir

## 1) Load Results JSON Files
Read the JSON summaries for each dataset and the combined table.

In [ ]:
json_files = {
    "all": base_dir / "phase1_all_datasets_results.json",
    "cifar10": base_dir / "phase1_cifar10_results.json",
    "cifar100": base_dir / "phase1_cifar100_results.json",
    "fashion_mnist": base_dir / "phase1_fashion_mnist_results.json",
}

missing = [k for k, p in json_files.items() if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing JSON files: {missing} in {base_dir}")

def load_json(path: Path):
    return json.loads(path.read_text())

data = {k: load_json(p) for k, p in json_files.items()}
list(data.keys())

## 2) Normalize and Merge Metrics into a DataFrame
Convert JSON structures into tidy tables for plotting.

In [ ]:
records = []
for ds_key in ["cifar10", "fashion_mnist", "cifar100"]:
    payload = data[ds_key]
    for exp in payload.get("experiments", []):
        cfg = exp.get("config", {})
        name = exp.get("experiment_name", "")
        if "higher_lr" in name:
            variant = "higher_lr"
        elif "sgd" in name:
            variant = "sgd"
        elif "seed" in name:
            variant = "baseline"
        else:
            variant = "other"

        records.append(
            {
                "dataset": ds_key,
                "experiment": name,
                "variant": variant,
                "optimizer": cfg.get("optimizer"),
                "lr": cfg.get("lr"),
                "seed": cfg.get("seed"),
                "final_train_acc": exp.get("final_train_acc"),
                "final_val_acc": exp.get("final_val_acc"),
                "test_acc": exp.get("test_acc"),
                "test_loss": exp.get("test_loss"),
                "params": exp.get("params"),
                "mean_epoch_time": exp.get("mean_epoch_time"),
            }
        )

df = pd.DataFrame.from_records(records)

summary = (
    df.groupby(["dataset", "variant"], as_index=False)
    .agg(
        test_acc_mean=("test_acc", "mean"),
        test_acc_std=("test_acc", "std"),
        final_val_acc_mean=("final_val_acc", "mean"),
        test_loss_mean=("test_loss", "mean"),
        n=("test_acc", "size"),
    )
    .sort_values(["dataset", "variant"])
)

summary.style.format(
    {
        "test_acc_mean": "{:.4f}",
        "test_acc_std": "{:.4f}",
        "final_val_acc_mean": "{:.4f}",
        "test_loss_mean": "{:.4f}",
    }
)

## 3) Plot Accuracy vs. Epochs by Dataset
If per‑epoch metrics are present, plot train/val accuracy curves. Otherwise, show a distribution plot of final test accuracy by dataset/variant.

In [ ]:
def extract_epoch_rows(payload):
    rows = []
    for exp in payload.get("experiments", []):
        for key in ["epoch_metrics", "history", "metrics"]:
            if key in exp and isinstance(exp[key], list):
                for item in exp[key]:
                    row = {"experiment": exp.get("experiment_name"), "dataset": payload.get("dataset")}
                    row.update(item)
                    rows.append(row)
    return rows

epoch_rows = []
for ds_key in ["cifar10", "fashion_mnist", "cifar100"]:
    epoch_rows.extend(extract_epoch_rows(data[ds_key]))

epoch_df = pd.DataFrame(epoch_rows)

if epoch_df.empty:
    # Fallback: mean + points (clearer than violin for small N)
    plt.figure(figsize=(12, 7))
    order = ["cifar10", "fashion_mnist", "cifar100"]
    hue_order = ["baseline", "higher_lr", "sgd", "other"]

    sns.pointplot(
        data=df,
        x="dataset",
        y="test_acc",
        hue="variant",
        order=order,
        hue_order=hue_order,
        estimator="mean",
        errorbar="sd",
        markers="o",
        linestyles="-",
        dodge=0.4,
        palette="Set2",
    )
    sns.stripplot(
        data=df,
        x="dataset",
        y="test_acc",
        hue="variant",
        order=order,
        hue_order=hue_order,
        dodge=True,
        color="black",
        size=5,
        alpha=0.6,
        jitter=0.12,
    )

    plt.title("Final Test Accuracy: Mean ± SD with Individual Runs")
    plt.xlabel("Dataset")
    plt.ylabel("Test Accuracy")
    plt.grid(alpha=0.25)

    handles, labels = plt.gca().get_legend_handles_labels()
    unique = list(dict.fromkeys(labels))
    plt.legend(handles[: len(unique)], unique, title="Variant", bbox_to_anchor=(1.02, 1), loc="upper left")

    plt.tight_layout()
    acc_dist_path = output_dir / "final_test_accuracy_mean_points.png"
    plt.savefig(acc_dist_path, dpi=240)
    saved_files.append(acc_dist_path)
    plt.show()
else:
    acc_cols = [c for c in epoch_df.columns if "acc" in c]
    if not acc_cols:
        print("Per-epoch data found, but no accuracy columns detected.")
    else:
        # Prefer train/val naming if present
        candidates = [c for c in acc_cols if c in ("train_acc", "val_acc", "valid_acc")]
        if candidates:
            plot_cols = candidates
        else:
            plot_cols = acc_cols

        plot_df = epoch_df.melt(
            id_vars=[c for c in ["dataset", "epoch"] if c in epoch_df.columns],
            value_vars=plot_cols,
            var_name="metric",
            value_name="value",
        )
        if "epoch" not in plot_df.columns:
            plot_df["epoch"] = np.arange(len(plot_df))

        plt.figure(figsize=(12, 7))
        sns.lineplot(data=plot_df, x="epoch", y="value", hue="dataset", style="metric", markers=True)
        plt.title("Accuracy vs. Epochs by Dataset")
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.grid(alpha=0.25)
        plt.legend(title="Dataset / Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.tight_layout()
        acc_path = output_dir / "accuracy_vs_epochs.png"
        plt.savefig(acc_path, dpi=220)
        saved_files.append(acc_path)
        plt.show()

## 4) Plot Loss vs. Epochs by Dataset
If per‑epoch metrics are present, plot train/val loss curves. Otherwise, show a distribution plot of final test loss by dataset/variant.

In [ ]:
if epoch_df.empty:
    # Fallback: mean + points (clearer than violin for small N)
    plt.figure(figsize=(12, 7))
    order = ["cifar10", "fashion_mnist", "cifar100"]
    hue_order = ["baseline", "higher_lr", "sgd", "other"]

    sns.pointplot(
        data=df,
        x="dataset",
        y="test_loss",
        hue="variant",
        order=order,
        hue_order=hue_order,
        estimator="mean",
        errorbar="sd",
        markers="o",
        linestyles="-",
        dodge=0.4,
        palette="Set3",
    )
    sns.stripplot(
        data=df,
        x="dataset",
        y="test_loss",
        hue="variant",
        order=order,
        hue_order=hue_order,
        dodge=True,
        color="black",
        size=5,
        alpha=0.6,
        jitter=0.12,
    )

    plt.title("Final Test Loss: Mean ± SD with Individual Runs")
    plt.xlabel("Dataset")
    plt.ylabel("Test Loss")
    plt.grid(alpha=0.25)

    handles, labels = plt.gca().get_legend_handles_labels()
    unique = list(dict.fromkeys(labels))
    plt.legend(handles[: len(unique)], unique, title="Variant", bbox_to_anchor=(1.02, 1), loc="upper left")

    plt.tight_layout()
    loss_dist_path = output_dir / "final_test_loss_mean_points.png"
    plt.savefig(loss_dist_path, dpi=240)
    saved_files.append(loss_dist_path)
    plt.show()
else:
    loss_cols = [c for c in epoch_df.columns if "loss" in c]
    if not loss_cols:
        print("Per-epoch data found, but no loss columns detected.")
    else:
        candidates = [c for c in loss_cols if c in ("train_loss", "val_loss", "valid_loss")]
        if candidates:
            plot_cols = candidates
        else:
            plot_cols = loss_cols

        plot_df = epoch_df.melt(
            id_vars=[c for c in ["dataset", "epoch"] if c in epoch_df.columns],
            value_vars=plot_cols,
            var_name="metric",
            value_name="value",
        )
        if "epoch" not in plot_df.columns:
            plot_df["epoch"] = np.arange(len(plot_df))

        plt.figure(figsize=(12, 7))
        sns.lineplot(data=plot_df, x="epoch", y="value", hue="dataset", style="metric", markers=True)
        plt.title("Loss vs. Epochs by Dataset")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.grid(alpha=0.25)
        plt.legend(title="Dataset / Metric", bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.tight_layout()
        loss_path = output_dir / "loss_vs_epochs.png"
        plt.savefig(loss_path, dpi=220)
        saved_files.append(loss_path)
        plt.show()

## 5) Create Summary Bar Charts (Final Metrics)
Aggregate final metrics and compare dataset performance and variants.

In [ ]:
plot_df = summary.copy()
plot_df["test_acc_std"] = plot_df["test_acc_std"].fillna(0.0)

# Test Accuracy bar chart
plt.figure(figsize=(10, 6))
ax = sns.barplot(
    data=plot_df,
    x="dataset",
    y="test_acc_mean",
    hue="variant",
    palette="deep",
)

# Manual error bars using bar centers
for patch, (_, row) in zip(ax.patches, plot_df.iterrows()):
    x = patch.get_x() + patch.get_width() / 2
    ax.errorbar(
        x,
        row["test_acc_mean"],
        yerr=row["test_acc_std"],
        fmt="none",
        ecolor="black",
        capsize=4,
        lw=1,
        zorder=5,
    )

ax.set_title("Final Test Accuracy by Dataset and Variant")
ax.set_xlabel("Dataset")
ax.set_ylabel("Test Accuracy")
ax.legend(title="Variant", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
acc_bar_path = output_dir / "final_test_accuracy_bar.png"
plt.savefig(acc_bar_path, dpi=200)
saved_files.append(acc_bar_path)
plt.show()

# Test Loss bar chart
plt.figure(figsize=(10, 6))
ax = sns.barplot(
    data=plot_df,
    x="dataset",
    y="test_loss_mean",
    hue="variant",
    palette="muted",
)
ax.set_title("Final Test Loss by Dataset and Variant")
ax.set_xlabel("Dataset")
ax.set_ylabel("Test Loss")
ax.legend(title="Variant", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
loss_bar_path = output_dir / "final_test_loss_bar.png"
plt.savefig(loss_bar_path, dpi=200)
saved_files.append(loss_bar_path)
plt.show()

## 6) Save Figures to Disk
List all saved figure files.

In [ ]:
if saved_files:
    pd.DataFrame({"saved_plot": [str(p) for p in saved_files]})
else:
    print("No plots were saved. If per-epoch metrics are missing, only summary charts are produced.")